<a href="https://colab.research.google.com/github/azeemorg/Youtube_Comment_Sentiment_Analysis/blob/main/yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# YouTube Comment Sentiment Analysis

# Analyze YouTube comments using the YouTube Data API and
# multilingual BERT-based sentiment analysis.

In [ ]:
#Step-1 : EXTRACTING YOUTUBE COMMENTS USING YT API

In [ ]:
import googleapiclient.discovery
import googleapiclient.errors
import pandas as pd
from transformers import pipeline
import matplotlib.pyplot as plt
import re


import os

DEVELOPER_KEY = os.getenv("YOUTUBE_API_KEY")

if not DEVELOPER_KEY:
    raise ValueError("YOUTUBE_API_KEY environment variable is not set.")
#YouTube API setup
api_service_name = "youtube"
api_version = "v3"


youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY
)

# Get video ID input from the user
video_id = input("Enter the Video Id Here: ")
all_comments = []
next_page_token = None

# Fetch all comments from the video
while True:
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        pageToken=next_page_token
    )
    response = request.execute()

    for item in response['items']:
        comment_data = item['snippet']['topLevelComment']['snippet']
        all_comments.append([
            comment_data.get('authorDisplayName', None),
            comment_data.get('publishedAt', None),
            comment_data.get('updatedAt', None),
            comment_data.get('likeCount', 0),
            comment_data.get('textDisplay', None)
        ])

    next_page_token = response.get('nextPageToken')
    if not next_page_token:
        break


In [ ]:
# Step-2 : Framing the Comments in the Dataframe

In [ ]:
# Create a DataFrame from the collected comments
df = pd.DataFrame(all_comments, columns=['AuthorDisplayName', 'PublishedAt', 'UpdatedAt', 'LikeCount', 'text'])

print(f"Fetched {len(all_comments)} comments.")
print(df.head())

df.to_csv("Dataset.csv", index=False)

print("Comments exported to 'Dataset.csv'.")


In [ ]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

In [ ]:
import pandas as pd
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import emoji



# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Remove URLs
df['cleaned_text'] = df['text'].str.replace(r'http\S+|www\S+|https\S+', '', regex=True)

# Remove usernames, hashtags, and email addresses if necessary
df['cleaned_text'] = df['cleaned_text'].str.replace(r'@\S+|#\S+|\S+@\S+', '', regex=True)

# Convert to lowercase
df['cleaned_text'] = df['cleaned_text'].str.lower()

#Translating Emoji
df['cleaned_text'] = df['cleaned_text'].apply(lambda text: emoji.demojize(text))

# Remove special characters, numbers, and punctuation
df['cleaned_text'] = df['cleaned_text'].str.replace(r'[^a-z\s]', '', regex=True)

# Lemmatization to reduce words to their base form
df['cleaned_text'] = df['cleaned_text'].apply(lambda text: " ".join([lemmatizer.lemmatize(word) for word in text.split()]))

# Remove extra whitespace
df['cleaned_text'] = df['cleaned_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Remove missing or empty values
df = df.dropna(subset=['cleaned_text'])
df = df[df['cleaned_text'] != '']

stop_words = set(stopwords.words('english'))
df['cleaned_text'] = df['cleaned_text'].apply(lambda text: " ".join([word for word in text.split() if word not in stop_words]))

# Display cleaned text
print(df[['text', 'cleaned_text']].head())

df.to_csv('cleaned_comments.csv', index=False)



In [ ]:
# Initialize the sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# Define sentiment mapping
sentiment_mapping = {
    "1 star": "1 (Very Negative)",
    "2 stars": "2 (Negative)",
    "3 stars": "3 (Neutral)",
    "4 stars": "4 (Positive)",
    "5 stars": "5 (Very Positive)"
}

# Function to analyze sentiment
def analyze_sentiment(text):
    text = text[:512]  # Truncate text to avoid exceeding the model's token limit
    result = sentiment_pipeline(text)
    return sentiment_mapping[result[0]['label']]

# Apply sentiment analysis on the cleaned text
df['sentiment'] = df['text'].apply(analyze_sentiment)


sentiment_counts = df['sentiment'].value_counts()

#sentiment counts
print(sentiment_counts)

# Print the DataFrame with sentiment results
print(df.head())

# Save the DataFrame to a CSV file
df.to_csv("Dataset_with_Sentiment.csv", index=False)

In [ ]:
# Bar Graph
colors = ['red', 'green', 'blue', 'orange', 'purple']
df['sentiment'].value_counts().plot.bar(color=colors)
plt.title('Sentiment Distribution')
plt.grid(True)
plt.legend()
plt.xlabel('Sentiment')
plt.ylabel('No of Comments')
plt.plot()


In [ ]:
#Pie Chart
df['sentiment'].value_counts().plot.pie(autopct='%2.2f%%')
plt.title('Sentiment Propotion')
plt.ylabel('')

In [ ]:
#Wordcloud
from wordcloud import WordCloud, STOPWORDS
#Uniting all comments in words
text = df['cleaned_text'].str.cat(sep='')
text

#Set the stopwords
stopwords=set(STOPWORDS)
new_words = ['ref','referee']
new_stopwords=stopwords.union(new_words)


#Size of word cloud
plt.rcParams['figure.figsize'] = [10,10]

#Make Wordclouds
wordcloud=WordCloud(    width=1600,  # Increase resolution width
    height=800,  # Increase resolution height
    max_font_size=100,  # Set larger max font size
   background_color='white',stopwords=new_stopwords,colormap='flag').generate(text)

#Plot the wordcloud

plt.imshow(wordcloud,interpolation = 'bilinear')
plt.tight_layout(pad=0)  # Remove padding
plt.axis('off')
plt.plot()

In [ ]:
from sklearn.utils import resample


df_very_negative = df[df['sentiment'] == "1 (Very Negative)"]
df_negative = df[df['sentiment'] == "2 (Negative)"]
df_neutral = df[df['sentiment'] == "3 (Neutral)"]
df_positive = df[df['sentiment'] == "4 (Positive)"]
df_very_positive = df[df['sentiment'] == "5 (Very Positive)"]


print("Rows in df_very_positive:", len(df_very_positive))
print("Rows in df_very_negative:", len(df_very_negative))
print("Rows in df_negative:", len(df_negative))
print("Rows in df_neutral:", len(df_neutral))
print("Rows in df_positive:", len(df_positive))


# Get the Maximum number of samples in any class
n_samples = max(len(df_very_negative), len(df_negative), len(df_neutral), len(df_positive), len(df_very_positive))

# Performing resampling to balance the classes
df_very_negative_unsampled = resample(df_very_negative,
                                      replace=True,
                                      n_samples=n_samples,
                                      random_state=42)

df_negative_unsampled = resample(df_negative,
                                 replace=True,
                                 n_samples=n_samples,
                                 random_state=42)

df_neutral_unsampled = resample(df_neutral,
                                replace=True,
                                n_samples=n_samples,
                                random_state=42)

df_positive_unsampled = resample(df_positive,
                                 replace=True,
                                 n_samples=n_samples,
                                 random_state=42)

df_very_positive_unsampled = resample(df_very_positive,
                                      replace=True,
                                      n_samples=n_samples,
                                      random_state=42)

# Concatenate the resampled data
final_data = pd.concat([df_very_negative_unsampled, df_negative_unsampled, df_neutral_unsampled,
                        df_positive_unsampled, df_very_positive_unsampled])

print("Data After Balancing :")
print(final_data['sentiment'].value_counts())


In [ ]:
corpus = []
for text in final_data['text']:
    corpus.append(text)
corpus[0:10]

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=100)
X = cv.fit_transform(corpus).toarray()
Y = final_data.iloc[:, -1].values

In [27]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state = 0)
classifier =  GaussianNB()
classifier.fit(X_train, y_train)

GaussianNB()

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
import seaborn as sns
sns.heatmap(cm,cmap="Greens",annot= True,
           cbar_kws={"orientation":"vertical","label":"color_bar"})
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
nb_score = accuracy_score(y_test, y_pred)
print('Accuracy',nb_score)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))
